<a href="https://colab.research.google.com/github/sayan1506/Triagegeist/blob/main/Triagegegist_Phase_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

triagegeist_path = kagglehub.competition_download('triagegeist')

print('Data source import complete.')


100%|██████████| 8.96M/8.96M [00:00<00:00, 32.5MB/s]

Extracting files...


Data source import complete.


In [ ]:
import os
import pandas as pd

triagegeist_path = '/root/.cache/kagglehub/competitions/triagegeist/'

train = pd.read_csv(os.path.join(triagegeist_path, "train.csv"))
test = pd.read_csv(os.path.join(triagegeist_path, "test.csv"))
complaints = pd.read_csv(os.path.join(triagegeist_path, "chief_complaints.csv"))
patient_history = pd.read_csv(os.path.join(triagegeist_path, "patient_history.csv"))

# Fix complaints
complaints_clean = complaints[['patient_id', 'chief_complaint_raw']].rename(
    columns={'chief_complaint_raw': 'complaint_text'}
)

# Merge
train = train.merge(patient_history, on='patient_id', how='left')
train = train.merge(complaints_clean, on='patient_id', how='left')
test = test.merge(patient_history, on='patient_id', how='left')
test = test.merge(complaints_clean, on='patient_id', how='left')

# Safe drop
cols_to_drop = ['chief_complaint_system_x', 'chief_complaint_system_y', 'chief_complaint_system']
train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
test  = test.drop(columns=[c for c in cols_to_drop if c in test.columns])

# Target and features
TARGET = 'triage_acuity'
DROP_COLS = ['patient_id', 'site_id', 'triage_nurse_id',
             'disposition', 'ed_los_hours',
             'complaint_text', TARGET]

feature_cols = [c for c in train.columns if c not in DROP_COLS]

X_train = train[feature_cols]
y_train = train[TARGET]
X_test  = test[feature_cols]

train_text = train['complaint_text']
test_text  = test['complaint_text']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nTarget distribution:\n", y_train.value_counts().sort_index())
print("\nFeature count:", len(feature_cols))
print("\nSample complaints:\n", train_text.head())

X_train shape: (80000, 58)
X_test shape: (20000, 58)

Target distribution:
 triage_acuity
1     3222
2    13439
3    28921
4    23020
5    11398
Name: count, dtype: int64

Feature count: 58

Sample complaints:
 0    thunderclap headache, worsening with movement
1               contraception advice, intermittent
2            general health question, intermittent
3         erythema migrans tick bite, intermittent
4               cellulitis localised, intermittent
Name: complaint_text, dtype: object
